In [1]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

In [2]:
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

### 1. Chuẩn bị dữ liệu Đồ thị (Graph Data Preparation) cho PyG
Chuyển đổi DataFrame thành cấu trúc `edge_index` và `edge_attr` của PyTorch Geometric.

In [3]:
import torch
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data

print("--- BƯỚC 1: XỬ LÝ IP VÀ TẠO MAPPING ---")
all_ips = pd.concat([
    train_df['src_ip'], train_df['dst_ip'],
    valid_df['src_ip'], valid_df['dst_ip'],
    test_df['src_ip'], test_df['dst_ip']
]).unique()

ip_to_id = {ip: i for i, ip in enumerate(all_ips)}
num_nodes = len(all_ips)
print(f"Tổng số Nodes (IP duy nhất) trong toàn bộ dữ liệu: {num_nodes}")

def prepare_dynamic_graphs(df, ip_dict, window_size_sec=600):
    """Hàm cắt lược DataFrame thành danh sách các Đồ thị theo Cửa sổ thời gian (Time-Windows)"""
    # Xác định Group ID của cửa sổ thời gian
    min_time = df['timestamp_num'].min()
    # Tránh SettingWithCopyWarning
    df = df.copy()
    df['time_window'] = ((df['timestamp_num'] - min_time) // window_size_sec).astype(int)
    
    dynamic_graphs = []
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'time_window', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    
    # Nhóm theo từng cửa sổ thời gian và build đồ thị con (Snapshot)
    for window_id, group in df.groupby('time_window'):
        if len(group) == 0:
            continue
            
        src_nodes = group['src_ip'].map(ip_dict).values
        dst_nodes = group['dst_ip'].map(ip_dict).values
        edge_index = torch.tensor(np.vstack([src_nodes, dst_nodes]), dtype=torch.long)
        
        edge_features = group.drop(columns=cols_to_drop).values
        edge_attr = torch.tensor(edge_features, dtype=torch.float32)
        
        labels = torch.tensor(group['label'].values, dtype=torch.long)
        
        # Bọc thành đối tượng PyG Data cho 1 khoảng thời gian
        snapshot = Data(
            num_nodes=len(ip_dict), 
            edge_index=edge_index, 
            edge_attr=edge_attr, 
            y=labels
        )
        dynamic_graphs.append(snapshot)
        
    return dynamic_graphs

print("\n--- BƯỚC 2: CẮT ĐỒ THỊ THEO THỜI GIAN (TIME-WINDOW GRAPHS) ---")
WINDOW_SIZE = 600 # Nhóm từng khoảng 10 phút (600 giây) thành 1 đồ thị
print(f"Đang tiến hành cắt đồ thị với cửa sổ {WINDOW_SIZE} giây / snapshot...")

train_graphs = prepare_dynamic_graphs(train_df, ip_to_id, WINDOW_SIZE)
valid_graphs = prepare_dynamic_graphs(valid_df, ip_to_id, WINDOW_SIZE)
test_graphs = prepare_dynamic_graphs(test_df, ip_to_id, WINDOW_SIZE)

print(f"-> Tập Train được cắt thành {len(train_graphs):,} Snapshots (Đồ thị con)")
print(f"-> Tổng lưu lượng Edges trong Train: {sum([g.num_edges for g in train_graphs]):,}")

--- BƯỚC 1: XỬ LÝ IP VÀ TẠO MAPPING ---
Tổng số Nodes (IP duy nhất) trong toàn bộ dữ liệu: 23

--- BƯỚC 2: CẮT ĐỒ THỊ THEO THỜI GIAN (TIME-WINDOW GRAPHS) ---
Đang tiến hành cắt đồ thị với cửa sổ 600 giây / snapshot...
-> Tập Train được cắt thành 23 Snapshots (Đồ thị con)
-> Tổng lưu lượng Edges trong Train: 2,470,638


### 2. Kiểm tra Đặc tính đồ thị (Graph Diagnostic Checks)
Dùng PyTorch Geometric `Data` object để bọc dữ liệu và tiến hành check tính hợp lệ của đồ thị (Label Imbalance, NaNs/Infs, Self-loops...)

In [4]:
from torch_geometric.data import Data

print("--- BƯỚC 3: KIỂM TRA ĐẶC TÍNH CỦA ĐỒ THỊ ĐỘNG (DIAGNOSTIC CHECKS) ---")

# Lấy thử Snapshot đầu tiên và Snapshot bất kỳ có nhiều luồng nhất để phân tích
max_edge_graph = max(train_graphs, key=lambda g: g.num_edges)
first_graph = train_graphs[0]

print(f"1. Cấu trúc tổng quát của MỘT SỐ Đồ thị con (Snapshot lớn nhất):")
print(f"   - Số lượng Nút (IP): {max_edge_graph.num_nodes:,} (Khung IP cố định)")
print(f"   - Số lượng Cạnh (Flows): {max_edge_graph.num_edges:,}")
print(f"   - Đồ thị có chứa vòng lặp (Self-loops): {max_edge_graph.has_self_loops()}")
print(f"   - Đồ thị có Nút cô lập (Isolated Nodes)? {max_edge_graph.has_isolated_nodes()}")

print("\n2. Kiểm tra tính hợp lệ của Feature trên Snapshot đầu tiên:")
has_nan = torch.isnan(first_graph.edge_attr).any().item()
has_inf = torch.isinf(first_graph.edge_attr).any().item()
print(f"   - Có chứa giá trị NaN? {'CÓ LỖI' if has_nan else 'Không (An toàn)'}")
print(f"   - Có chứa giá trị Inf? {'CÓ LỖI' if has_inf else 'Không (An toàn)'}")

print("\n3. Kiểm tra tính Đa đồ thị (Multi-graph) trong Snapshot 10 phút nhỏ này:")
unique_edges = torch.unique(max_edge_graph.edge_index, dim=1).shape[1]
duplicate_edges = max_edge_graph.num_edges - unique_edges
print(f"   - Số cạnh trùng lặp trên cùng cặp IP: {duplicate_edges:,} / {max_edge_graph.num_edges:,} flows")
if duplicate_edges > 0:
    print("   -> Snapshot trong 10 phút vẫn là Multi-graph, nhưng mức độ Leak dữ liệu theo ngày đã được triệt tiêu!")
    print("   -> (Giải pháp E-GraphSAGE): Vẫn dùng LinkNeighborLoader thay vì Load gọn toàn bộ.")

print("\n4. Kiểm tra phân phối Nhãn (Class Imbalance) trên toàn bộ các Snapshot Train:")
total_label_counts = torch.zeros(11, dtype=torch.int64) # Giả sử có tối đa 11 classes, điều chỉnh số liệu nếu cần
total_edges = sum([g.num_edges for g in train_graphs])

for g in train_graphs:
    counts = torch.bincount(g.y, minlength=11)
    total_label_counts += counts

for i, count in enumerate(total_label_counts[:11]):
    if count > 0:
        percentage = (count.item() / total_edges) * 100
        print(f"   - Lớp {i}: {count.item():>9,} flows ({percentage:>6.2f}%)")

--- BƯỚC 3: KIỂM TRA ĐẶC TÍNH CỦA ĐỒ THỊ ĐỘNG (DIAGNOSTIC CHECKS) ---
1. Cấu trúc tổng quát của MỘT SỐ Đồ thị con (Snapshot lớn nhất):
   - Số lượng Nút (IP): 23 (Khung IP cố định)
   - Số lượng Cạnh (Flows): 689,408
   - Đồ thị có chứa vòng lặp (Self-loops): False
   - Đồ thị có Nút cô lập (Isolated Nodes)? True

2. Kiểm tra tính hợp lệ của Feature trên Snapshot đầu tiên:
   - Có chứa giá trị NaN? Không (An toàn)
   - Có chứa giá trị Inf? Không (An toàn)

3. Kiểm tra tính Đa đồ thị (Multi-graph) trong Snapshot 10 phút nhỏ này:
   - Số cạnh trùng lặp trên cùng cặp IP: 689,387 / 689,408 flows
   -> Snapshot trong 10 phút vẫn là Multi-graph, nhưng mức độ Leak dữ liệu theo ngày đã được triệt tiêu!
   -> (Giải pháp E-GraphSAGE): Vẫn dùng LinkNeighborLoader thay vì Load gọn toàn bộ.

4. Kiểm tra phân phối Nhãn (Class Imbalance) trên toàn bộ các Snapshot Train:
   - Lớp 0:    64,488 flows (  2.61%)
   - Lớp 1: 1,573,220 flows ( 63.68%)
   - Lớp 2:     8,164 flows (  0.33%)
   - Lớp 3:   117,

In [7]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing

# 1. ĐỊNH NGHĨA LỚP MESSAGE PASSING KẾT HỢP CẠNH (EDGE-ENHANCED SAGE)
class EdgeSAGEConv(MessagePassing):
    def __init__(self, in_channels, out_channels, edge_channels):
        super().__init__(aggr='mean') 
        
        self.msg_mlp = nn.Sequential(
            nn.Linear(in_channels + edge_channels, out_channels),
            nn.LayerNorm(out_channels), # ĐỘT PHÁ 1: Định chuẩn Batch để chặn Gradients vô cực
            nn.ReLU(),
        )
        
        self.update_mlp = nn.Sequential(
            nn.Linear(in_channels + out_channels, out_channels),
            nn.LayerNorm(out_channels), # ĐỘT PHÁ 1: Định chuẩn Batch để chặn Gradients vô cực
            nn.ReLU()
        )

    def forward(self, x, edge_index, edge_attr):
        return self.propagate(edge_index, x=x, edge_attr=edge_attr)

    def message(self, x_j, edge_attr):
        msg_input = torch.cat([x_j, edge_attr], dim=-1)
        return self.msg_mlp(msg_input)

    def update(self, aggr_out, x):
        update_input = torch.cat([x, aggr_out], dim=-1)
        return self.update_mlp(update_input)

# 2. ĐỊNH NGHĨA MODEL TỔNG THỂ (CHỐNG NAN & EXPLODING GRADIENT)
class EGraphSAGE(nn.Module):
    def __init__(self, num_nodes, node_emb_dim, edge_in_dim, hidden_dim, num_classes):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, node_emb_dim)
        
        # ĐỘT PHÁ 2: Chuẩn hoá Scale luồng mạng để không vượt quá sức chịu đựng của Float
        self.edge_norm = nn.LayerNorm(edge_in_dim) 
        
        self.conv1 = EdgeSAGEConv(node_emb_dim, hidden_dim, edge_in_dim)
        self.conv2 = EdgeSAGEConv(hidden_dim, hidden_dim, edge_in_dim)
        
        classifier_dim = hidden_dim * 2 + edge_in_dim
        self.classifier = nn.Sequential(
            nn.Linear(classifier_dim, hidden_dim * 2),
            nn.LayerNorm(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def encode_context(self, edge_index, edge_attr):
        """Hàm đóng gói: Xoá Rác (NaN/Inf) và Xây dựng Bối cảnh"""
        # Tự động thay thế RÁC ẨN (chia cho số 0, NaN ẩn) sinh ra trong Dữ liệu
        clean_attr = torch.nan_to_num(edge_attr, nan=0.0, posinf=0.0, neginf=0.0)
        clean_attr = self.edge_norm(clean_attr)
        
        x = self.node_emb.weight
        x = F.relu(self.conv1(x, edge_index, clean_attr))
        x = self.conv2(x, edge_index, clean_attr)
        return x, clean_attr

    def predict_edges(self, x_context, clean_attr, edge_index, query_idx):
        """Hàm đóng gói: Rút đúng mini-batch ném vào Classifier"""
        src_nodes = edge_index[0, query_idx]
        dst_nodes = edge_index[1, query_idx]
        batch_attr = clean_attr[query_idx]
        
        edge_repr = torch.cat([x_context[src_nodes], x_context[dst_nodes], batch_attr], dim=-1)
        return self.classifier(edge_repr)

print("Đã xây dựng xong Mô Hình E-GraphSAGE (Phiên bản Chống NaN)!")

Đã xây dựng xong Mô Hình E-GraphSAGE (Phiên bản Chống NaN)!


In [9]:
import torch.optim as optim

print("--- BƯỚC 4: KHỞI TẠO MÔ HÌNH VÀ THUẬT TOÁN TỐI ƯU ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

# Trích xuất số chiều từ đồ thị đầu tiên
node_emb_dim = 64
hidden_dim = 128
edge_in_dim = train_graphs[0].edge_attr.shape[1]

# Giả định dataset có 11 nhãn (hoặc bạn có thể tính max classes)
num_classes = 11

model = EGraphSAGE(
    num_nodes=len(ip_to_id),
    node_emb_dim=node_emb_dim,
    edge_in_dim=edge_in_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("Khởi tạo hoàn tất! Mô hình đã sẵn sàng huấn luyện.")

--- BƯỚC 4: KHỞI TẠO MÔ HÌNH VÀ THUẬT TOÁN TỐI ƯU ---
Đang sử dụng thiết bị: cuda
Khởi tạo hoàn tất! Mô hình đã sẵn sàng huấn luyện.


In [11]:
import copy
import time
import random
import numpy as np
from sklearn.metrics import classification_report

print("--- BƯỚC 5: HUẤN LUYỆN, EARLY STOPPING & ĐÁNH GIÁ (ANTI-NAN GRAPHS) ---")

def evaluate_graphs(model, graphs, criterion, device):
    """Hàm đánh giá Cấu trúc mới"""
    model.eval()
    total_loss = 0
    total_edges = sum(g.num_edges for g in graphs)
    all_preds = []
    all_trues = []
    
    with torch.no_grad():
        for g in graphs:
            if g.num_edges == 0:
                continue
                
            g_edge_idx = g.edge_index.to(device)
            g_edge_attr = g.edge_attr.to(device)
            g_y = g.y.to(device)
            
            # GỘP 1: Dọn rác và Xây Context
            x_ctx, clean_attr = model.encode_context(g_edge_idx, g_edge_attr)
            
            # GỘP 2: Chia mẻ nhỏ Predict Tránh bể RAM
            for i in range(0, g.num_edges, 100000): 
                idx = torch.arange(i, min(i+100000, g.num_edges), device=device)
                
                val_out = model.predict_edges(x_ctx, clean_attr, g_edge_idx, idx)
                loss = criterion(val_out, g_y[idx])
                
                total_loss += loss.item() * len(idx)
                all_preds.append(torch.argmax(val_out, dim=1).cpu().numpy())
                all_trues.append(g_y[idx].cpu().numpy())
                
    avg_loss = total_loss / (total_edges if total_edges > 0 else 1)
    
    if len(all_preds) > 0:
        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_trues)
        macro_f1 = classification_report(y_true, y_pred, output_dict=True, zero_division=0)['macro avg']['f1-score']
    else:
        macro_f1, y_pred, y_true = 0.0, [], []
        
    return avg_loss, macro_f1, y_true, y_pred


# THIẾT LẬP THAM SỐ
epochs = 30
batch_size = 16384 
patience = 5  

best_val_f1 = -1.0
best_model_wts = copy.deepcopy(model.state_dict())
patience_counter = 0
total_train_edges = sum(g.num_edges for g in train_graphs)

print(f"\n=> Bắt đầu huấn luyện {epochs} Epochs với Cấu Trúc Kháng Nhiễu...")
start_time = time.time()

for epoch in range(epochs):
    model.train()
    total_train_loss = 0
    
    random.shuffle(train_graphs)
    
    for graph_idx, g in enumerate(train_graphs):
        if g.num_edges == 0:
            continue
            
        g_edge_idx = g.edge_index.to(device)
        g_edge_attr = g.edge_attr.to(device)
        g_y = g.y.to(device)
        
        perm = torch.randperm(g.num_edges, device=device)
        
        for i in range(0, g.num_edges, batch_size):
            idx = perm[i:i+batch_size]
            
            optimizer.zero_grad()
            
            # CHẠY VÒNG ĐỜI KÈM CƠ CHẾ ANTI-NAN NGAY TRONG MODEL
            x_ctx, clean_attr = model.encode_context(g_edge_idx, g_edge_attr)
            preds = model.predict_edges(x_ctx, clean_attr, g_edge_idx, idx)
            
            loss = criterion(preds, g_y[idx])
            loss.backward()
            
            # CẮT ĐẠO HÀM DƯ: Đề phòng Outlier
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            total_train_loss += loss.item() * len(idx)
            
    avg_train_loss = total_train_loss / total_train_edges
    
    # 3. Validation
    val_loss, val_f1, _, _ = evaluate_graphs(model, valid_graphs, criterion, device)
    
    print(f"Epoch {epoch+1:>2}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro-F1: {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_wts = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"   -> Epoch {epoch+1}: Kỷ lục Validation Macro-F1 mới ({val_f1:.4f})! Đã lưu trọng số.")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n=> EARLY STOPPING kích hoạt ở Epoch {epoch+1} (Valid F1 ngừng cải thiện sau {patience} epochs).")
            break

print(f"\nQuá trình Training hoàn tất trong {time.time() - start_time:.2f} giây.")

print("\n--- BƯỚC 6: ĐÁNH GIÁ TRÊN TẬP TEST TEST (VỚI MODEL TỐT NHẤT) ---")
model.load_state_dict(best_model_wts)
test_loss, test_f1, y_test_true, y_test_pred = evaluate_graphs(model, test_graphs, criterion, device)

print(f"Test Loss: {test_loss:.4f} | Test Macro-F1: {test_f1:.4f}\n")
print(classification_report(y_test_true, y_test_pred, digits=4, zero_division=0))

--- BƯỚC 5: HUẤN LUYỆN, EARLY STOPPING & ĐÁNH GIÁ (ANTI-NAN GRAPHS) ---

=> Bắt đầu huấn luyện 30 Epochs với Cấu Trúc Kháng Nhiễu...
Epoch  1/30 | Train Loss: 2.0555 | Val Loss: 2.6488 | Val Macro-F1: 0.0225
   -> Epoch 1: Kỷ lục Validation Macro-F1 mới (0.0225)! Đã lưu trọng số.
Epoch  2/30 | Train Loss: 1.4081 | Val Loss: 3.6430 | Val Macro-F1: 0.0225


KeyboardInterrupt: 